In [ ]:
# This code compares penalized fitness strategy without novelty with 
# the one with novelty component (different novelty weights)

from src.controller.ga import GeneticAlgorithm, GAConfig
from src.model.molecule import Molecule
from src.model.population import Population
from src.model.fitness import (
    compute_fitness,
    compute_fitness_penalized,
    novelty_augmented_fitness,
)
from src.view.viewer import population_grid
from src.view.plots import plot_multiple_fitness_histories
from IPython.display import display
from src.model.fitness import archive


# initial molecules in the digital soup
soup = ['[C][#N]', '[C][=O]', '[C][O]', '[C][C][O]', '[C][C][=O]',
         '[O][=C][C][O]', '[O][=C][O]', '[N][C][=Branch1][C][=O][N]', 
         '[N]', '[O]', '[N][C][C][=Branch1][C][=O][O]', '[C][C][Branch1][=Branch1][C][=Branch1][C][=O][O][N]', 
         '[C][C][=Branch1][C][=O][O]', '[C][C][N]', '[C][S]', '[C][C][=Branch1][C][=O][C][=Branch1][C][=O][O]', 
         '[C][C][=Branch1][C][=O][C]', '[O][=C][=O]', '[O][=C][=S]', '[O][P][=Branch1][C][=O][Branch1][C][O][O]', 
         '[C][=C][C][=C][C][=C][Ring1][=Branch1]', '[C][=C][N][=C][NH1][Ring1][Branch1]', '[C][C][=C][NH1][C][=Ring1][Branch1]', 
         '[C][C][C][C][C][Ring1][Branch1]', '[C][C][C][C][C][C][Ring1][=Branch1]', '[N][C][=N][C][=C][N][Ring1][Branch1]', 
         '[C][C][=C][O][C][=Ring1][Branch1]', '[O][C][C][=Branch1][C][=O][O]', '[C][=N][C][=N][C][NH1][C][=N][C][Ring1][=Branch2][=Ring1][Branch1]', 
         '[O][P][=Branch1][C][=O][Branch1][C][O][O][P][=Branch1][C][=O][Branch1][C][O][O]', '[C][C][Branch1][C][O][C][=Branch1][C][=O][O]', 
         '[O][=C][C][Branch1][C][O][C][O]', '[O][=C][Branch1][Ring1][C][O][C][O]', '[N][C][=O]', '[C][=C]', '[C][C][C][=Branch1][C][=O][O]', 
         '[O][=C][Branch1][C][O][C][C][C][=Branch1][C][=O][O]', '[N][C][C][S]', '[N][C][=Branch1][C][=S][N]', 
         '[O][C][C@H1][O][C][Branch1][C][O][C@H1][Branch1][C][O][C@@H1][Ring1][#Branch1][O]']

initial = [Molecule(s) for s in soup]

def make_initial_population():
    # fresh population for every run
    return Population(initial)

# multiple runs with different fitness functions

histories = []
labels = []

fitness_runs = [
    ("Band-penalized fitness", compute_fitness_penalized),
    ("Novelty-augmented w=0.1",
     lambda mol: novelty_augmented_fitness(mol, novelty_weight=0.1)),
    ("Novelty-augmented w=0.5",
     lambda mol: novelty_augmented_fitness(mol, novelty_weight=0.5))
]

for label, fitness_fn in fitness_runs:
    # reset novelty archive memory
    archive.archive.clear()
    print(f"\n=== Running GA with {label} ===")
    pop = make_initial_population()

    # selection and replacement parameters
    cfg = GAConfig(
        mu=100,
        lam=100,
        mutation_rate=1,
        crossover_rate=1,
        tournament_k=2,
        rank_bias=1.7,
        random_seed=0,
    )

    # run the GA
    ga = GeneticAlgorithm(cfg, fitness_fn)
    history = ga.evolve(pop, generations=50)
    print("Evolution done!")

    histories.append(history)
    labels.append(label)

    # showing the last generation molecules for each run
    print(f"\nLast generation for {label}")
    last_pop = history[-1]
    for n in last_pop.molecules:
        print(n.smiles)

    display(population_grid(last_pop, n=30, subimg_size=(400, 400)))

# one joint plot for all strategies
plot_multiple_fitness_histories(histories, labels)